<a href="https://colab.research.google.com/github/sathushetty7/RAG-PDF-Chatbot/blob/main/PDF_RAG_Question_Answering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q pypdf langchain langchain-community langchain-text-splitters sentence-transformers faiss-cpu transformers

In [3]:
import os
import torch

from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

import faiss
import numpy as np

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.11.0+cpu
CUDA available: False


In [4]:
from google.colab import files

uploaded = files.upload()

Saving pdf-Attention Is All You Need.pdf to pdf-Attention Is All You Need.pdf


In [6]:
pdf_path = next(iter(uploaded))

print("PDF path:", pdf_path)

PDF path: pdf-Attention Is All You Need.pdf


In [7]:
from pypdf import PdfReader

reader = PdfReader(pdf_path)

text = ""

for page in reader.pages:
    page_text = page.extract_text()

    if page_text:
        text += page_text + "\n"

print("Number of pages:", len(reader.pages))
print("Characters extracted:", len(text))

print("\n--- FIRST 2000 CHARACTERS ---\n")
print(text[:2000])

Number of pages: 15
Characters extracted: 39511

--- FIRST 2000 CHARACTERS ---

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on a

In [60]:
# SPLIT PDF TEXT INTO CHUNKS

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create smaller overlapping chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

# Split PDF text into chunks
chunks = text_splitter.split_text(text)

print("Number of chunks:", len(chunks))
print("\n--- FIRST CHUNK ---\n")
print(chunks[0])

Number of chunks: 60

--- FIRST CHUNK ---

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best


In [61]:
# Load the embedding model
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Convert chunks into embeddings
embeddings = embedding_model.encode(
    chunks,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embedding shape: (60, 384)


In [62]:
#  BUILD FAISS VECTOR INDEX

import faiss
import numpy as np

# Convert embeddings to float32
embedding_matrix = np.array(
    embeddings
).astype("float32")

# Normalize embeddings for cosine similarity
faiss.normalize_L2(embedding_matrix)

# Create cosine similarity index
index = faiss.IndexFlatIP(
    embedding_matrix.shape[1]
)

# Add embeddings to the index
index.add(embedding_matrix)

print("Total vectors in index:", index.ntotal)

Total vectors in index: 60


In [65]:
# LOAD FLAN-T5 MODEL

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load FLAN-T5 tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "google/flan-t5-base"
)

# Load FLAN-T5 model
model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-base"
)

print("FLAN-T5 loaded successfully.")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


FLAN-T5 loaded successfully.


In [69]:
# INSPECT RETRIEVED CHUNKS

question = input("Ask a question about the PDF: ")

# Convert question into an embedding
query_embedding = embedding_model.encode(
    [question]
).astype("float32")

# Normalize for cosine similarity
faiss.normalize_L2(query_embedding)

# Retrieve the top 5 chunks
similarities, indices = index.search(
    query_embedding,
    5
)

print("\n__________RETRIEVED CHUNKS_____________")

for i, idx in enumerate(indices[0]):
    print(
        f"\n--- Chunk {i + 1} | "
        f"Similarity: {similarities[0][i]:.4f} ---"
    )
    print(chunks[idx])

Ask a question about the PDF: What is the Transformer architecture?

__________RETRIEVED CHUNKS_____________

--- Chunk 1 | Similarity: 0.4742 ---
Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
3.1 Encoder and Decoder Stacks
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-
wise fully connected feed-forward network. We employ a residual connection [11] around each of
the two sub-layers, followed by layer normalization [ 1]. That is, the output of each sub-layer is
LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer

--- Chunk 2 | Similarity: 0.4275 ---
translation quality after being trained for a

In [67]:
# QUESTION ANSWERING WITH RAG

def ask_question(query, top_k=5):

    # Convert question into an embedding
    query_embedding = embedding_model.encode(
        [query]
    ).astype("float32")

    # Normalize for cosine similarity
    faiss.normalize_L2(query_embedding)

    # Retrieve relevant chunks
    similarities, indices = index.search(
        query_embedding,
        top_k
    )

    # Get retrieved chunks
    retrieved_chunks = [
        chunks[idx] for idx in indices[0]
    ]

    # Include the first chunk for broad PDF questions
    broad_words = [
        "mainly",
        "about",
        "paper",
        "document",
        "summary",
        "overview",
        "topic"
    ]

    if any(word in query.lower() for word in broad_words):
        retrieved_chunks.insert(0, chunks[0])

    # Combine chunks into context
    context = "\n\n".join(retrieved_chunks)

    # Create prompt
    prompt = f"""
Answer the question using only the information in the context.

Give a clear and concise answer.
If the answer is not present in the context, say:
"I could not find the answer in the PDF."

Context:
{context}

Question:
{query}

Answer:
"""

    # Tokenize the prompt
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    # Generate answer
    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

    # Convert generated tokens into text
    result_text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return result_text

In [71]:
question = input("Ask a question about the PDF: ")

answer = ask_question(question)

print("\n__________ANSWER_____________")
print(answer)

Ask a question about the PDF: What is this PDF mainly about?

__________ANSWER_____________
Attention Is All You Need
